# Analysis for Covid-19 Effects on GDP/Employment

## Setup

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


df = pd.read_csv("../data/clean_wdi_data.csv")
df

,country_code,country_name,year,gdp_per_capita,gdp_growth_pct,employment_ratio,shock_period_flag
0,BGD,Bangladesh,1990,473.44,5.62,NaN,Normal
1,BGD,Bangladesh,1991,480.67,3.49,55.67,Normal
2,BGD,Bangladesh,1992,497.37,5.44,55.80,Normal
3,BGD,Bangladesh,1993,511.19,4.71,55.90,Normal
4,BGD,Bangladesh,1994,521.32,3.89,56.02,Normal
...,...,...,...,...,...,...,...
233,PAK,Pakistan,2019,1573.83,2.50,49.10,Normal
234,PAK,Pakistan,2020,1526.01,-1.27,48.49,Covid
235,PAK,Pakistan,2021,1595.03,6.51,49.39,Normal
236,PAK,Pakistan,2022,1642.28,4.78,49.63,Normal


## Crisis Period

In [14]:
df_2020 = df[(df["year"] >= 2017) & (df["year"] <= 2023)].copy()

# Classify periods
def classify_period(year):
    if year in [2017, 2018, 2019]:
        return "Pre-Crisis"
    elif year in [2020]:
        return "Crisis"
    elif year in [2021, 2022, 2023]:
        return "Post-Crisis"
    return np.nan

df_2020["period"] = df_2020["year"].apply(classify_period)

print(df_2020.head())
print("\nYears included:", sorted(df_2020["year"].unique()))
print("Countries included:", sorted(df_2020["country_name"].unique()))

   country_code country_name  year  gdp_per_capita  gdp_growth_pct  \
27          BGD   Bangladesh  2017         1373.75            6.59   
28          BGD   Bangladesh  2018         1462.25            7.32   
29          BGD   Bangladesh  2019         1564.21            7.88   
30          BGD   Bangladesh  2020         1604.67            3.45   
31          BGD   Bangladesh  2021         1702.08            6.94   

    employment_ratio shock_period_flag       period  
27             55.78            Normal   Pre-Crisis  
28             56.30            Normal   Pre-Crisis  
29             56.79            Normal   Pre-Crisis  
30             56.46             Covid       Crisis  
31             57.75            Normal  Post-Crisis  

Years included: [2017, 2018, 2019, 2020, 2021, 2022, 2023]
Countries included: ['Bangladesh', 'Bhutan', 'India', 'Maldives', 'Nepal', 'Pakistan', 'Sri Lanka']


## Summary Statistics

In [8]:
print("Overall Summary Statistics (2017–2023)")
overall_summary = df_2020[["gdp_per_capita", "gdp_growth_pct", "employment_ratio"]].describe().round(2)
print(overall_summary)

print("\n Summary Statistics by Country")
country_summary = (
    df_2020.groupby("country_name")[["gdp_per_capita", "gdp_growth_pct", "employment_ratio"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
)
print(country_summary)

print("\nSummary Statistics by Period ")
period_summary = (
    df_2020.groupby("period")[["gdp_per_capita", "gdp_growth_pct", "employment_ratio"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
)
print(period_summary)

Overall Summary Statistics (2017–2023)
       gdp_per_capita  gdp_growth_pct  employment_ratio
count           49.00           49.00             49.00
mean          3434.86            4.06             51.51
std           3041.83            8.53              8.61
min            951.86          -32.91             34.25
25%           1564.21            2.31             48.47
50%           1936.03            5.21             49.81
75%           4033.15            6.94             58.38
max          11439.54           37.51             65.78

 Summary Statistics by Country
             gdp_per_capita                                        \
                       mean    median      std      min       max   
country_name                                                        
Bangladesh          1628.10   1604.67   182.37  1373.75   1885.38   
Bhutan              3301.19   3300.55   143.09  3092.12   3480.51   
India               1964.15   1936.03   170.80  1788.70   2270.91   
Maldives   

## Average values by country and period

In [15]:
country_period_avg = (
    df_2020.groupby(["country_name", "period"])[["gdp_per_capita", "gdp_growth_pct", "employment_ratio"]]
    .mean()
    .round(2)
    .reset_index()
)

print("Country and Period averages")
print(country_period_avg)

Country and Period averages
   country_name       period  gdp_per_capita  gdp_growth_pct  employment_ratio
0    Bangladesh       Crisis         1604.67            3.45             56.46
1    Bangladesh  Post-Crisis         1797.27            6.61             58.54
2    Bangladesh   Pre-Crisis         1466.74            7.26             56.29
3        Bhutan       Crisis         3092.12          -10.22             64.42
4        Bhutan  Post-Crisis         3345.48            4.75             62.69
5        Bhutan   Pre-Crisis         3326.58            3.78             61.98
6         India       Crisis         1806.50           -5.78             47.50
7         India  Post-Crisis         2111.48            8.83             50.86
8         India   Pre-Crisis         1869.36            5.71             48.67
9      Maldives       Crisis         7295.60          -32.91             55.14
10     Maldives  Post-Crisis        10712.92           18.76             59.42
11     Maldives   Pre-Cr

## GDP Growth 

In [23]:
fig_growth = px.line(
    df_2020,
    x="year",
    y="gdp_growth_pct",
    color="country_name",
    markers=True,
    title="Interactive GDP Growth Trends Around the 2020 Crisis",
    labels={
        "year": "Year",
        "gdp_growth_pct": "GDP Growth (%)",
        "country_name": "Country"
    }
)

fig_growth.add_vrect(
    x0=2020, x1=2021,
    opacity=0.2,
    line_width=0,
    annotation_text="Crisis",
    annotation_position="top left"
)

fig_growth.show()

## Employment ratio

In [22]:
fig_emp = px.line(
    df_2020,
    x="year",
    y="employment_ratio",
    color="country_name",
    markers=True,
    title="Interactive Employment Ratio Trends Around the 2020 Crisis",
    labels={
        "year": "Year",
        "employment_ratio": "Employment Ratio (%)",
        "country_name": "Country"
    }
)

fig_emp.add_vrect(
    x0=2020, x1=2021,
    opacity=0.2,
    line_width=0,
    annotation_text="Crisis",
    annotation_position="top left"
)

fig_emp.show()

## GDP per capita 

In [21]:
fig_gdppc = go.Figure()

for country in sorted(df_2020["country_name"].unique()):
    temp = df_2020[df_2020["country_name"] == country]
    fig_gdppc.add_trace(
        go.Scatter(
            x=temp["year"],
            y=temp["gdp_per_capita"],
            mode="lines+markers",
            name=country,
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Year: %{x}<br>"
                "GDP per Capita: %{y:,.2f}<extra></extra>"
            )
        )
    )

fig_gdppc.add_vrect(
    x0=2020, x1=2021,
    fillcolor="lightblue",
    opacity=0.25,
    line_width=0,
    annotation_text="2020 Crisis",
    annotation_position="top left"
)

fig_gdppc.update_layout(
    title="Interactive GDP per Capita in South Asia Around the 2020 Financial Crisis",
    xaxis_title="Year",
    yaxis_title="GDP per Capita (constant 2015 US$)",
    xaxis=dict(dtick=1),
    hovermode="x unified",
    legend_title="Country"
)

fig_gdppc.show()

## Average by period

In [24]:
fig_bar = px.bar(
    country_period_avg,
    x="country_name",
    y="gdp_growth_pct",
    color="period",
    barmode="group",
    title="Interactive Average GDP Growth by Country and Period",
    labels={
        "country_name": "Country",
        "gdp_growth_pct": "Average GDP Growth (%)",
        "period": "Period"
    }
)

fig_bar.show()